# 📊 Metrics

**Measure your LLM application performance**

## 📋 Overview

**What you'll learn:**
- Key metrics for LLM apps
- Prometheus integration
- Custom metrics
- Performance tracking
- Cost tracking

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

In [ ]:
from prometheus_client import Counter, Histogram, Gauge, Summary
import time

# Define metrics
llm_requests_total = Counter(
    'llm_requests_total',
    'Total LLM API requests',
    ['model', 'status']
)

llm_latency = Histogram(
    'llm_latency_seconds',
    'LLM request latency',
    ['model']
)

llm_tokens = Counter(
    'llm_tokens_total',
    'Total tokens used',
    ['model', 'type']  # type: prompt/completion
)

llm_cost = Counter(
    'llm_cost_dollars',
    'Total cost in dollars',
    ['model']
)

print('✅ Metrics defined')

## 📊 Key LLM Metrics

### 1. Request Metrics
```python
# Track requests
llm_requests_total.labels(model='gpt-4', status='success').inc()
llm_requests_total.labels(model='gpt-4', status='error').inc()
```

### 2. Latency Metrics
```python
start = time.time()
response = call_llm()
llm_latency.labels(model='gpt-4').observe(time.time() - start)
```

### 3. Token & Cost Metrics
```python
llm_tokens.labels(model='gpt-4', type='prompt').inc(prompt_tokens)
llm_tokens.labels(model='gpt-4', type='completion').inc(completion_tokens)
llm_cost.labels(model='gpt-4').inc(cost)
```

### 4. Cache Metrics
```python
cache_hits = Counter('cache_hits_total', 'Cache hits')
cache_misses = Counter('cache_misses_total', 'Cache misses')
cache_hit_rate = Gauge('cache_hit_rate', 'Cache hit rate percentage')
```

## 🎯 Metrics Wrapper

```python
class MetricsWrapper:
    def __init__(self, client):
        self.client = client
    
    def chat_completion(self, **kwargs):
        model = kwargs.get('model', 'gpt-3.5-turbo')
        
        # Start timer
        start = time.time()
        
        try:
            # Call LLM
            response = self.client.chat.completions.create(**kwargs)
            
            # Record success
            llm_requests_total.labels(
                model=model,
                status='success'
            ).inc()
            
            # Record latency
            latency = time.time() - start
            llm_latency.labels(model=model).observe(latency)
            
            # Record tokens
            usage = response.usage
            llm_tokens.labels(model=model, type='prompt').inc(
                usage.prompt_tokens
            )
            llm_tokens.labels(model=model, type='completion').inc(
                usage.completion_tokens
            )
            
            # Calculate cost
            cost = calculate_cost(model, usage)
            llm_cost.labels(model=model).inc(cost)
            
            return response
            
        except Exception as e:
            # Record error
            llm_requests_total.labels(
                model=model,
                status='error'
            ).inc()
            raise
```

## ✅ Summary

**Essential LLM metrics:**
- **Requests**: Total, success rate, error rate
- **Latency**: P50, P95, P99
- **Tokens**: Prompt, completion, total
- **Cost**: Per model, per user, total
- **Cache**: Hit rate, savings

**Best practices:**
- Track all LLM calls
- Use labels for filtering (model, user, endpoint)
- Monitor costs in real-time
- Set up alerts on anomalies

### Next: `11_observability/03_tracing.ipynb`